In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
import traceback # Import traceback for detailed error printing

In [18]:
# Imports
from TSB_AD.model_wrapper import (run_Unsupervise_AD, run_Semisupervise_AD)
from TSB_AD.evaluation.metrics import get_metrics
from TSB_AD.HP_list import Optimal_Uni_algo_HP_dict
from TSB_AD.utils.slidingWindows import find_length_rank
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [19]:
Unsupervise_AD_Pool = ['FFT', 'SR', 'Sub_IForest', 'IForest', 'LOF', 'Sub_LOF', 'POLY', 'MatrixProfile', 'Sub_PCA', 'PCA', 'HBOS', 
                        'Sub_HBOS', 'KNN', 'Sub_KNN','KMeansAD', 'KMeansAD_U', 'KShapeAD', 'COPOD', 'CBLOF', 'COF', 'EIF', 'RobustPCA']
Semisupervise_AD_Pool = ['Left_STAMPi', 'SAND', 'MCD', 'Sub_MCD', 'OCSVM', 'Sub_OCSVM', 'AutoEncoder', 'CNN', 'LSTMAD', 'TranAD', 'USAD', 'OmniAnomaly', 
                        'AnomalyTransformer', 'TimesNet', 'FITS', 'Donut', 'M2N2']


In [21]:
# --- Data Loading and Preparation ---
data_path = '/Users/kai/Documents/Time_Series_Anomaly_Detection_Seminar/Time-Series-Anomaly-Detection-Seminar/TSB-AD/Datasets/TSB-AD-U/001_NAB_id_1_Facility_tr_1007_1st_2014.csv'
filename = os.path.basename(data_path)
print(f"Processing file: {filename}")
df = pd.read_csv(data_path).dropna()

if df.empty:
    raise ValueError("Dataframe is empty after loading and dropna.")

data = df.iloc[:, 0:-1].values.astype(float)
label = df['Label'].astype(int).to_numpy()
print(f"Original Data shape: {data.shape}, Label shape: {label.shape}") # Add shape print

if data.size == 0 or label.size == 0 or data.shape[0] != label.shape[0]:
    raise ValueError("Data or label array is empty or shapes mismatch.")

# --- Create data_train ---
# (Your existing data_train logic is fine)
try:
    train_index_str = filename.split('.')[0].split('_')[-3]
    if train_index_str == 'tr' and filename.split('.')[0].split('_')[-2].isdigit():
         train_index = int(filename.split('.')[0].split('_')[-2])
         print(f"Using train_index derived from filename: {train_index}")
    elif train_index_str.isdigit():
         train_index = int(train_index_str)
         print(f"Using train_index derived from filename (fallback position): {train_index}")
    else:
        raise ValueError("Filename does not match expected 'tr_INDEX' pattern.")

    if train_index <= 0 or train_index >= len(data):
         raise ValueError(f"Derived train_index {train_index} is out of bounds for data length {len(data)}.")
    data_train = data[:train_index, :]
except (IndexError, ValueError) as e:
    print(f"Warning: Could not derive train_index from filename '{filename}' ({e}). Falling back to 50% split.")
    split_point = int(len(data) * 0.5)
    if split_point == 0 and len(data) > 0:
        split_point = 1
    print(f"Using fallback train split: first {split_point} points.")
    data_train = data[:split_point, :]

if data_train.shape[0] == 0:
     raise ValueError("Training data segment (data_train) is empty.")
print(f"Data shape: {data.shape}, Training data shape: {data_train.shape}")

# --- Calculate Sliding Window ONCE ---
slidingWindow = find_length_rank(data, rank=1)
print(f"Calculated sliding window: {slidingWindow}")

# --- Define run_metrics Function (More Robust) ---
def run_metrics(output, true_labels, sliding_window):
    """
    Calculates evaluation metrics for a given model output. Handles errors.
    """
    # 1. Check initial output type
    if not isinstance(output, np.ndarray):
        if isinstance(output, str): # Model likely returned an error string
            print(f"  Model execution returned an error string: {output}")
            return {'error': output}
        else: # Handle other unexpected output types
            print(f"  Model execution returned unexpected type: {type(output)}. Skipping metrics.")
            return {'error': f'Unexpected output type: {type(output)}'}

    # 2. Check output shape and length BEFORE scaling
    print(f"  Received output shape: {output.shape}, Label shape: {true_labels.shape}") # Debug print
    if output.ndim > 2 or (output.ndim == 2 and output.shape[1] != 1):
         print(f"  Warning: Output array has unexpected shape {output.shape}. Attempting to use first column if possible.")
         if output.ndim == 2 and output.shape[1] > 1:
             output = output[:, 0].copy() # Try using only the first column
             print(f"  Using first column only. New output shape: {output.shape}")
         else:
             return {'error': f'Unsupported output array shape: {output.shape}'}

    if output.shape[0] != len(true_labels):
         print(f"  ERROR: Output length ({output.shape[0]}) != label length ({len(true_labels)}). Skipping metrics.")
         return {'error': f'Length mismatch: Output {output.shape[0]} vs Label {len(true_labels)}'}

    # 3. Check for NaNs/Infs in scores BEFORE scaling
    if np.any(np.isnan(output)) or np.any(np.isinf(output)):
        print("  Warning: Output contains NaN or Inf values. Attempting to replace with 0.")
        output = np.nan_to_num(output, nan=0.0, posinf=0.0, neginf=0.0) # Replace with 0, adjust if needed

    # 4. Scale the scores
    output_scaled = None
    try:
        # Reshape to 2D column vector for scaler, then flatten back
        output_scaled = MinMaxScaler(feature_range=(0,1)).fit_transform(output.reshape(-1, 1)).ravel()
        if np.any(np.isnan(output_scaled)): # Check again after scaling
             raise ValueError("NaNs found after scaling.")
    except Exception as scale_e:
         print(f"  Error during scaling: {scale_e}. Skipping metrics.")
         return {'error': f'Scaling failed: {scale_e}'}

    # 5. Calculate threshold robustly
    threshold = 0.0 # Default threshold
    try:
        mean_score = np.mean(output_scaled)
        std_dev = np.std(output_scaled)

        if std_dev < 1e-8: # Check if standard deviation is effectively zero
            print("  Warning: Scores have near-zero standard deviation.")
            # Handle constant scores: maybe threshold slightly above the constant value?
            # Or simply use the mean, results might be poor. Using mean for now.
            threshold = mean_score
        else:
            threshold = mean_score + 3 * std_dev

        threshold = np.clip(threshold, 0, 1) # Ensure threshold is valid [0, 1]
        print(f"  Using threshold {threshold:.4f}")

    except Exception as thresh_e:
        print(f"  Error calculating threshold: {thresh_e}. Using default 0.0. Metrics might be affected.")
        # Continue, but threshold might be suboptimal

    # 6. Get binary predictions
    pred = (output_scaled >= threshold).astype(int) # Use >= for threshold inclusivity

    # 7. Calculate Metrics with warning suppression for UndefinedMetric
    evaluation_result = None
    try:
        # Temporarily suppress the UndefinedMetricWarning about precision
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning, message="Precision is ill-defined")
            warnings.filterwarnings("ignore", category=RuntimeWarning) # Suppress numpy runtime warnings too if desired
            evaluation_result = get_metrics(score=output_scaled, labels=true_labels, slidingWindow=sliding_window, pred=pred)
        return evaluation_result
    except Exception as metric_e:
        print(f"  Error during get_metrics: {metric_e}")
        # traceback.print_exc() # Uncomment for full traceback if needed
        return {'error': f'get_metrics failed: {metric_e}'}


# --- Initialize Results Dictionaries ---
unsupervised_results = {}
semi_supervised_results = {}


# --- Run Unsupervised Models ---
print("\n--- Running Unsupervised Models ---")
for model in Unsupervise_AD_Pool:
    print(f"\nRunning {model}...") # Add newline for spacing
    try:
        model_hp = Optimal_Uni_algo_HP_dict.get(model, {})
        if not model_hp: print(f"  Note: No specific HPs found for {model}.")
        output = run_Unsupervise_AD(model, data, **model_hp)
        unsupervised_results[model] = run_metrics(output, label, slidingWindow)
    except Exception as e:
        print(f"!!!! CRITICAL Error running model {model}: {e} !!!!") # Distinguish from caught errors
        traceback.print_exc()
        unsupervised_results[model] = {'error': f'CRITICAL Execution failed: {e}'}

print("\n--- Unsupervised Results ---")
for model, result in unsupervised_results.items():
    print(f"\nModel: {model}")
    if isinstance(result, dict) and 'error' not in result: # Check it's a dict and not error
        for metric, value in result.items():
            # Check for NaN values before printing
            if pd.isna(value):
                 print(f"  {metric}: NaN")
            else:
                 print(f"  {metric}: {value:.4f}")
    else:
        print(f"  Result: {result}") # Print error dict or other non-result types


# --- Run Semi-supervised Models ---
print("\n--- Running Semi-supervised Models ---")
for model in Semisupervise_AD_Pool:
    print(f"\nRunning {model}...") # Add newline for spacing
    try:
        model_hp = Optimal_Uni_algo_HP_dict.get(model, {})
        if not model_hp: print(f"  Note: No specific HPs found for {model}.")
        output = run_Semisupervise_AD(model, data_train, data, **model_hp)
        semi_supervised_results[model] = run_metrics(output, label, slidingWindow)
    except Exception as e:
        print(f"!!!! CRITICAL Error running model {model}: {e} !!!!") # Distinguish from caught errors
        traceback.print_exc()
        semi_supervised_results[model] = {'error': f'CRITICAL Execution failed: {e}'}


print("\n--- Semi-supervised Results ---")
for model, result in semi_supervised_results.items():
    print(f"\nModel: {model}")
    if isinstance(result, dict) and 'error' not in result: # Check it's a dict and not error
        for metric, value in result.items():
             # Check for NaN values before printing
            if pd.isna(value):
                 print(f"  {metric}: NaN")
            else:
                 print(f"  {metric}: {value:.4f}")
    else:
        print(f"  Result: {result}") # Print error dict or other non-result types


# Combine results into a DataFrame

rows = []

for model, metrics in unsupervised_results.items():
    row = metrics if metrics else {'error': 'N/A'}
    row['model'] = f'{model}_unsup'
    rows.append(row)

for model, metrics in semi_supervised_results.items():
    row = metrics if metrics else {'error': 'N/A'}
    row['model'] = f'{model}_semi'
    rows.append(row)

results_df = pd.DataFrame(rows)
cols = ['model'] + [col for col in results_df.columns if col != 'model']
results_df = results_df[cols]

print("\n--- Combined Results DataFrame ---")
print(results_df)
results_df.to_csv(f'{filename}_all_models_results.csv')


Processing file: 001_NAB_id_1_Facility_tr_1007_1st_2014.csv
Original Data shape: (4031, 1), Label shape: (4031,)
Using train_index derived from filename (fallback position): 1007
Data shape: (4031, 1), Training data shape: (1007, 1)
Calculated sliding window: 6

--- Running Unsupervised Models ---

Running FFT...
  Note: No specific HPs found for FFT.
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.8606

Running SR...
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.1082

Running Sub_IForest...
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.3357

Running IForest...
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.7893

Running LOF...
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.0639

Running Sub_LOF...
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.2104

Running POLY...
  Received output shape: (4031,), Label shape: (4031,)

/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



Running Sub_HBOS...
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 1.0000

Running KNN...
  Note: No specific HPs found for KNN.
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.0000

Running Sub_KNN...
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.8923

Running KMeansAD...
  Note: No specific HPs found for KMeansAD.
Required padding_length=0
Reversing window-based scores to point-based scores:
Before reverse-windowing: scores.shape=(4012,)
After reverse-windowing: scores.shape=(4031,)
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 1.0000

Running KMeansAD_U...
Required padding_length=0
Reversing window-based scores to point-based scores:
Before reverse-windowing: scores.shape=(4020,)
After reverse-windowing: scores.shape=(4031,)
  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 1.0000

Running KShapeAD...
  Received output shape: (4031,), Label shape: (403

/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (8). Possibly due

  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.0954

Running EIF...
  Note: No specific HPs found for EIF.


...

  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.0702

Running Donut...
----- GPU is unavailable -----
----- Using CPU -----
An error occurred while running the model 'run_Donut': stack expects a non-empty TensorList
  Model execution returned an error string: An error occurred while running the model 'run_Donut': stack expects a non-empty TensorList

Running M2N2...
  Note: No specific HPs found for M2N2.
----- GPU is unavailable -----
----- Using CPU -----
======================TRAIN MODE======================


calculating reconstruction errors: 100%|██████████| 1/1 [00:00<00:00, 507.78it/s]


======================TEST MODE======================


inference: 100%|██████████| 3/3 [00:00<00:00, 748.49it/s]


  Received output shape: (4031,), Label shape: (4031,)
  Using threshold 0.3781

--- Semi-supervised Results ---

Model: Left_STAMPi
  AUC-PR: 0.2679
  AUC-ROC: 0.8176
  VUS-PR: 0.2481
  VUS-ROC: 0.7797
  Standard-F1: 0.0000
  PA-F1: 0.0000
  Event-based-F1: 0.0000
  R-based-F1: 0.0000
  Affiliation-F: NaN

Model: SAND
  AUC-PR: 0.0822
  AUC-ROC: 0.5148
  VUS-PR: 0.0841
  VUS-ROC: 0.5198
  Standard-F1: 0.0000
  PA-F1: 0.0000
  Event-based-F1: 0.0000
  R-based-F1: 0.0000
  Affiliation-F: 0.1203

Model: MCD
  Result: {'error': "An error occurred while running the model 'run_MCD': unsupported operand type(s) for +: 'NoneType' and 'float'"}

Model: Sub_MCD
  AUC-PR: 0.1296
  AUC-ROC: 0.5793
  VUS-PR: 0.1290
  VUS-ROC: 0.5842
  Standard-F1: 0.0878
  PA-F1: 0.6955
  Event-based-F1: 0.3830
  R-based-F1: 0.1618
  Affiliation-F: 0.7925

Model: OCSVM
  AUC-PR: 0.0851
  AUC-ROC: 0.5000
  VUS-PR: 0.0864
  VUS-ROC: 0.5003
  Standard-F1: 0.1568
  PA-F1: 0.1568
  Event-based-F1: 0.1568
  R-based-F1: 

In [16]:

results_df

,model,AUC-PR,AUC-ROC,VUS-PR,VUS-ROC,Standard-F1,PA-F1,Event-based-F1,R-based-F1,Affiliation-F,error
0,FFT_unsup,0.676853,0.841954,0.595102,0.836680,0.297767,0.358852,0.500000,0.437500,0.496498,NaN
1,SR_unsup,0.140060,0.516936,0.134766,0.522197,0.088643,0.997093,0.941176,0.344670,0.959480,NaN
2,Sub_IForest_unsup,0.151817,0.492654,0.146034,0.498979,0.111702,0.982808,0.777778,0.224515,0.872507,NaN
3,IForest_unsup,0.154150,0.699186,0.154945,0.704111,0.005556,0.543611,0.100000,0.063361,0.506985,NaN
4,LOF_unsup,0.139951,0.503918,0.138900,0.511673,0.067416,0.998544,0.960000,0.347096,0.946797,NaN
5,Sub_LOF_unsup,0.259450,0.568544,0.254805,0.572924,0.189974,1.000000,1.000000,0.453878,0.959841,NaN
6,POLY_unsup,0.268692,0.605159,0.261176,0.610934,0.125683,1.000000,1.000000,0.416458,0.955618,NaN
7,MatrixProfile_unsup,0.093572,0.518498,0.093966,0.522927,0.016484,0.344037,0.200000,0.097211,0.832503,NaN
8,Sub_PCA_unsup,0.188552,0.505614,0.178645,0.511267,0.130790,1.000000,1.000000,0.418950,0.955810,NaN
9,PCA_unsup,0.795651,0.905929,0.795044,0.908464,0.614141,0.757246,0.800000,0.681806,0.793643,NaN
